# 08 · CNN de Imágenes — Wildfire Prediction Dataset (Transfer Learning)

**Etapa 4** — modelo de imágenes con CNN en PyTorch/GPU (cronograma original), usando
**transfer learning** sobre ResNet18 preentrenada en vez de entrenar desde cero — mucho
más rápido y con mejor desempeño con relativamente pocas imágenes.

Clasifica imágenes satelitales en `wildfire` / `nowildfire`.

**Entrada:** `DATOS_BASE_DIR/data/wildfire_images/{train,valid,test}/{wildfire,nowildfire}/*.jpg`
**Salidas:** `modelos/cnn_wildfire.pt`, `resultados/tablas/metricas_cnn.json`,
`resultados/figuras/cnn_*.png`


In [ ]:
import os, json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # tolera imagenes truncadas/corruptas del dataset de Kaggle
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

USUARIO = os.path.basename(os.path.expanduser("~"))
DATOS_BASE_DIR = os.path.join("/data", USUARIO, "deteccion_incendios")
BASE_DIR = os.path.join(os.path.expanduser("~"), "deteccion_incendios", "modelado")

DIR_IMAGENES = os.path.join(DATOS_BASE_DIR, "data", "wildfire_images")
DIR_MODELOS  = os.path.join(DATOS_BASE_DIR, "modelos")
DIR_TABLAS   = os.path.join(BASE_DIR, "resultados/tablas")
DIR_FIGURAS  = os.path.join(BASE_DIR, "resultados/figuras")
os.makedirs(DIR_MODELOS, exist_ok=True)
os.makedirs(DIR_TABLAS, exist_ok=True)
os.makedirs(DIR_FIGURAS, exist_ok=True)

SEMILLA = 42
torch.manual_seed(SEMILLA)

# --- MODO DE EJECUCION --------------------------------------------------
# "dev"  -> usa una fraccion chica de imagenes, corre rapido para probar el pipeline.
# "full" -> usa el dataset completo.
MODO = "dev"
FRACCION_DEV = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

assert os.path.isdir(DIR_IMAGENES), f"No se encontro {DIR_IMAGENES}. Ajusta DIR_IMAGENES si tu carpeta se llama distinto."


## 1. Transformaciones y carga de datos

Aumento de datos (flip, rotación leve) solo en `train`. `valid` se usa como set de validación durante el entrenamiento; `test` queda aparte para la evaluación final.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 256 if DEVICE.type == "cuda" else 32   # antes: 64 -- GPU con espacio de sobra
NUM_WORKERS = os.cpu_count()                          # antes: min(4, os.cpu_count()) -- usa todos los cores

tf_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # normalizacion estandar de ImageNet (requerida por ResNet preentrenada)
])
tf_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ds_train_full = datasets.ImageFolder(os.path.join(DIR_IMAGENES, "train"), transform=tf_train)
ds_val_full   = datasets.ImageFolder(os.path.join(DIR_IMAGENES, "valid"), transform=tf_eval)
ds_test_full  = datasets.ImageFolder(os.path.join(DIR_IMAGENES, "test"), transform=tf_eval)

print("Clases:", ds_train_full.classes)
print(f"Disponibles -> train={len(ds_train_full)}  valid={len(ds_val_full)}  test={len(ds_test_full)}")

def submuestrear(ds, fraccion, semilla):
    if fraccion >= 1.0:
        return ds
    n = int(len(ds) * fraccion)
    idx = np.random.RandomState(semilla).choice(len(ds), size=n, replace=False)
    return Subset(ds, idx)

if MODO == "dev":
    ds_train = submuestrear(ds_train_full, FRACCION_DEV, SEMILLA)
    ds_val   = submuestrear(ds_val_full, FRACCION_DEV, SEMILLA)
    ds_test  = submuestrear(ds_test_full, FRACCION_DEV, SEMILLA)
    print(f"Modo dev ({FRACCION_DEV*100:.0f}%) -> train={len(ds_train)}  valid={len(ds_val)}  test={len(ds_test)}")
else:
    ds_train, ds_val, ds_test = ds_train_full, ds_val_full, ds_test_full

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"), persistent_workers=True)
dl_val   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"), persistent_workers=True)
dl_test  = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"), persistent_workers=True)


## 2. Modelo: ResNet18 preentrenada (transfer learning)

Se congelan todas las capas convolucionales (ya saben detectar bordes/texturas/formas de
ImageNet) y solo se entrena la última capa, adaptada a nuestras 2 clases. Esto reduce
drásticamente el tiempo de entrenamiento y la cantidad de datos necesarios comparado con
entrenar una CNN desde cero.

> **Nota sobre internet en el nodo de cómputo:** descargar los pesos preentrenados
> requiere internet. Si el nodo de GPU no tiene salida a internet (nos pasó con `pip`
> antes), corre esta celda **una vez desde el nodo de login** para descargar y cachear
> los pesos (~45MB, no afecta la cuota), y luego sí funciona sin internet en el nodo GPU:
> ```bash
> python3 -c "from torchvision.models import resnet18, ResNet18_Weights; resnet18(weights=ResNet18_Weights.DEFAULT)"
> ```


In [ ]:
try:
    pesos = models.ResNet18_Weights.DEFAULT
    modelo_base = models.resnet18(weights=pesos)
    print("Pesos preentrenados de ImageNet cargados correctamente.")
except Exception as e:
    print(f"No se pudieron descargar los pesos preentrenados ({e}).")
    print("Entrenando desde cero (sin transfer learning) -> va a necesitar mas epocas/datos para converger bien.")
    print("Sugerencia: pre-descarga los pesos desde el nodo de LOGIN (ver celda de arriba) y reintenta.")
    modelo_base = models.resnet18(weights=None)

# Congelar todas las capas convolucionales
for param in modelo_base.parameters():
    param.requires_grad = False

# Reemplazar la ultima capa (clasificador) por una nueva, entrenable, para 2 clases
n_clases = len(ds_train_full.classes)
modelo_base.fc = nn.Sequential(
    nn.Linear(modelo_base.fc.in_features, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, n_clases),
)

modelo = modelo_base.to(DEVICE)
n_entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
n_totales = sum(p.numel() for p in modelo.parameters())
print(f"Parametros entrenables: {n_entrenables:,} de {n_totales:,} totales ({100*n_entrenables/n_totales:.1f}%)")


## 3. Monitor de recursos (mismo patrón que el pipeline tabular)

In [ ]:
import threading, subprocess

class MonitorRecursos:
    def __init__(self, intervalo=0.5):
        self.intervalo = intervalo
        self._corriendo = False
        self._hilo = None
        self.muestras_ram_mb = []
        self.muestras_gpu_mem_mb = []
        self.muestras_gpu_util_pct = []

    def _leer_gpu(self):
        try:
            salida = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
                 "--format=csv,noheader,nounits"], timeout=2
            ).decode().strip().split("\n")[0].split(",")
            return float(salida[0].strip()), float(salida[1].strip())
        except Exception:
            return None, None

    def _loop(self):
        try:
            import psutil
            proceso = psutil.Process(os.getpid())
        except ImportError:
            proceso = None
        while self._corriendo:
            if proceso is not None:
                self.muestras_ram_mb.append(proceso.memory_info().rss / 1e6)
            gpu_mem, gpu_util = self._leer_gpu()
            if gpu_mem is not None:
                self.muestras_gpu_mem_mb.append(gpu_mem)
                self.muestras_gpu_util_pct.append(gpu_util)
            threading.Event().wait(self.intervalo)

    def __enter__(self):
        self._corriendo = True
        self._hilo = threading.Thread(target=self._loop, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, *exc):
        self._corriendo = False
        self._hilo.join(timeout=2)

    def resumen(self):
        def _pico_prom(lista):
            return (max(lista), sum(lista) / len(lista)) if lista else (None, None)
        ram_pico, _ = _pico_prom(self.muestras_ram_mb)
        gpu_mem_pico, _ = _pico_prom(self.muestras_gpu_mem_mb)
        _, gpu_util_prom = _pico_prom(self.muestras_gpu_util_pct)
        return {"ram_pico_mb": ram_pico, "gpu_mem_pico_mb": gpu_mem_pico, "gpu_util_promedio_pct": gpu_util_prom}


## 4. Entrenamiento

In [ ]:
EPOCHS = 10
LR = 1e-3

criterio = nn.CrossEntropyLoss()
# Solo se optimizan los parametros entrenables (la nueva capa final)
optimizador = torch.optim.Adam(filter(lambda p: p.requires_grad, modelo.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode="min", factor=0.5, patience=2)

def correr_epoca(dl, entrenar):
    modelo.train(entrenar)
    perdida_total, correctos, n = 0.0, 0, 0
    for x, y in dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        if entrenar:
            optimizador.zero_grad()
        with torch.set_grad_enabled(entrenar):
            salida = modelo(x)
            perdida = criterio(salida, y)
        if entrenar:
            perdida.backward()
            optimizador.step()
        perdida_total += perdida.item() * x.size(0)
        correctos += (salida.argmax(1) == y).sum().item()
        n += x.size(0)
    return perdida_total / n, correctos / n

historial = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
mejor_val_loss = float("inf")
t0 = time.perf_counter()

with MonitorRecursos(intervalo=0.5) as mon:
    for epoca in range(1, EPOCHS + 1):
        tr_loss, tr_acc = correr_epoca(dl_train, entrenar=True)
        val_loss, val_acc = correr_epoca(dl_val, entrenar=False)
        scheduler.step(val_loss)

        historial["train_loss"].append(tr_loss); historial["train_acc"].append(tr_acc)
        historial["val_loss"].append(val_loss);   historial["val_acc"].append(val_acc)

        marcador = ""
        if val_loss < mejor_val_loss:
            mejor_val_loss = val_loss
            torch.save(modelo.state_dict(), os.path.join(DIR_MODELOS, "cnn_wildfire.pt"))
            marcador = " (mejor modelo guardado)"

        print(f"Epoca {epoca:>2}/{EPOCHS} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}{marcador}")

tiempo_entrenamiento = time.perf_counter() - t0
recursos = mon.resumen()
print(f"\nTiempo total de entrenamiento: {tiempo_entrenamiento:.1f}s ({DEVICE})")
if recursos["ram_pico_mb"]:
    print(f"RAM pico: {recursos['ram_pico_mb']:.0f} MB")
if recursos["gpu_mem_pico_mb"]:
    print(f"GPU mem pico: {recursos['gpu_mem_pico_mb']:.0f} MB | GPU util promedio: {recursos['gpu_util_promedio_pct']:.0f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].plot(historial["train_loss"], label="train"); axes[0].plot(historial["val_loss"], label="val")
axes[0].set_title("Perdida"); axes[0].set_xlabel("Epoca"); axes[0].legend()
axes[1].plot(historial["train_acc"], label="train"); axes[1].plot(historial["val_acc"], label="val")
axes[1].set_title("Exactitud"); axes[1].set_xlabel("Epoca"); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "cnn_curvas_entrenamiento.png"), dpi=150)
plt.show()


## 5. Evaluación sobre el test set

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)

modelo.load_state_dict(torch.load(os.path.join(DIR_MODELOS, "cnn_wildfire.pt")))
modelo.eval()

y_true, y_pred, y_score = [], [], []
t0 = time.perf_counter()
with torch.no_grad():
    for x, y in dl_test:
        x = x.to(DEVICE)
        salida = modelo(x)
        probs = torch.softmax(salida, dim=1)[:, 1]
        y_true.extend(y.numpy())
        y_pred.extend(salida.argmax(1).cpu().numpy())
        y_score.extend(probs.cpu().numpy())
tiempo_inferencia = time.perf_counter() - t0

y_true, y_pred, y_score = np.array(y_true), np.array(y_pred), np.array(y_score)

metricas = {
    "modelo": "CNN (ResNet18 transfer learning)",
    "backend": f"pytorch/{DEVICE.type}",
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred)),
    "recall": float(recall_score(y_true, y_pred)),
    "f1": float(f1_score(y_true, y_pred)),
    "roc_auc": float(roc_auc_score(y_true, y_score)),
    "tiempo_entrenamiento_s": tiempo_entrenamiento,
    "tiempo_inferencia_s": tiempo_inferencia,
    "epocas": EPOCHS,
    "n_train": len(ds_train),
    "n_test": len(ds_test),
    "clases": ds_train_full.classes,
    "ram_pico_mb": recursos["ram_pico_mb"],
    "gpu_mem_pico_mb": recursos["gpu_mem_pico_mb"],
    "gpu_util_promedio_pct": recursos["gpu_util_promedio_pct"],
}
print(json.dumps(metricas, indent=2, ensure_ascii=False))
print("\n", classification_report(y_true, y_pred, target_names=ds_train_full.classes))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,5))
ConfusionMatrixDisplay(cm, display_labels=ds_train_full.classes).plot(ax=ax, colorbar=False)
ax.set_title("CNN (ResNet18) — Matriz de confusion")
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "cnn_matriz_confusion.png"), dpi=150)
plt.show()


## 6. Guardar métricas

In [ ]:
with open(os.path.join(DIR_TABLAS, "metricas_cnn.json"), "w", encoding="utf-8") as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)
print("Metricas guardadas en:", os.path.join(DIR_TABLAS, "metricas_cnn.json"))
print("Modelo guardado en:", os.path.join(DIR_MODELOS, "cnn_wildfire.pt"))


## 7. Notas para el informe

- **Transfer learning vs. entrenar desde cero:** solo se entrena la última capa (clasificador),
  aprovechando que ResNet18 ya "sabe ver" formas/texturas/bordes de su preentrenamiento en
  ImageNet. Esto reduce drásticamente el tiempo de entrenamiento y la cantidad de datos
  necesarios comparado con una CNN entrenada desde cero.
- **Este modelo no es directamente comparable en F1/accuracy** con Random Forest/XGBoost/Red
  Neuronal tabular — trabaja sobre imágenes satelitales completas, no sobre los puntos de
  calor individuales de FIRMS. Sí es comparable en tiempo de cómputo GPU, útil para el
  análisis de rendimiento de la Etapa 5.
- Para la corrida final (no solo desarrollo), cambiar `MODO = "full"` en la primera celda.
